# 第8章: ニューラルネット

第7章で取り組んだポジネガ分類を題材として、ニューラルネットワークで分類モデルを実装する。なお、この章ではPyTorchやTensorFlow、JAXなどの深層学習フレームワークを活用せよ。

In [1]:
from dotenv import load_dotenv
import gensim
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

load_dotenv()
MODEL_PATH= os.getenv('WORD2VEC_MODEL_PATH')
SST2_PATH = os.getenv('SST2_PATH')

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

device: cuda


In [3]:
w2v = gensim.models.KeyedVectors.load_word2vec_format(MODEL_PATH, binary=True)

### 確認

In [12]:
print(type(w2v[0]))
print(w2v[0].shape)
print(np.zeros(w2v[0].shape))
tmp = torch.tensor([0], dtype=torch.float32)
print(tmp)
print(token_to_idx.get("aaaaaaaaaaaaaaaaaaaaaaaaaaa", 0) == 0)
tmptmp = torch.tensor([tmp, tmp])
print(tmptmp)


<class 'numpy.ndarray'>
(300,)
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
tensor([0.])
True
tensor([0., 0.])


## 70. 単語埋め込みの読み込み

事前学習済み単語埋め込みを活用し、$|V| \times d_\text{emb}$ の単語埋め込み行列$\pmb{E}$を作成せよ。ここで、$|V|$は単語埋め込みの語彙数、$d_\text{emb}$は単語埋め込みの次元数である。ただし、単語埋め込み行列の先頭の行ベクトル$\pmb{E}_{0,:}$は、将来的にパディング（`<PAD>`）トークンの埋め込みベクトルとして用いたいので、ゼロベクトルとして予約せよ。ゆえに、$\pmb{E}$の2行目以降に事前学習済み単語埋め込みを読み込むことになる。

もし、Google Newsデータセットの[学習済み単語ベクトル](https://drive.google.com/file/d/0B7XkCwpI5KDYNlNUTTlSS21pQmM/edit?usp=sharing)（300万単語・フレーズ、300次元）を全て読み込んだ場合、$|V|=3000001, d_\text{emb}=300$になるはずである（ただ、300万単語の中には、殆ど用いられない稀な単語も含まれるので、語彙を削減した方がメモリの節約になる）。

また、単語埋め込み行列の構築と同時に、単語埋め込み行列の各行のインデックス番号（トークンID）と、単語（トークン）への双方向の対応付けを保持せよ。

In [5]:
vector_size = w2v.vector_size
words = w2v.index_to_key

EMB = np.zeros((len(words)+1, vector_size), dtype="float32")

EMB[1:] = w2v.get_normed_vectors()

idx_to_token = {0:"<PAD>"}
token_to_idx = {"<PAD>":0}
for i, word in enumerate(words, 1):
    idx_to_token[i] = word
    token_to_idx[word] = i

print(EMB.shape)
print(len(idx_to_token))
print(len(token_to_idx))
print("=============")
print(EMB[923])
print(idx_to_token[923])
print(token_to_idx["Japan"])

(3000001, 300)
3000001
3000001
[ 1.83295589e-02  9.02378261e-02  6.02760501e-02  3.04905158e-02
 -9.58776921e-02 -2.78468300e-02 -1.42759066e-02 -1.07862405e-01
  2.90805493e-02  3.40154320e-02 -5.49886748e-02  4.51189131e-02
  9.23527777e-02 -1.21962063e-01 -3.54253985e-02 -5.46361841e-02
 -3.34866927e-03  6.83833510e-02  7.01458082e-02 -2.12596450e-03
  9.09428075e-02 -1.32184317e-02 -3.19004804e-02 -2.00038925e-02
 -3.52491513e-02  5.01198869e-04 -5.71036264e-02 -3.08430078e-03
 -5.71036264e-02  4.49426658e-02 -7.38029077e-04 -5.21687418e-02
 -9.44677219e-02 -6.73258752e-02 -1.29540628e-02 -4.49426658e-02
 -7.40232170e-02  5.57377189e-03  9.72876549e-02  1.96514018e-02
  2.29119491e-02  1.33065544e-02  2.34406851e-02  5.63986413e-02
  4.89963219e-02  1.19847111e-01 -1.22490805e-02  7.29657412e-02
  1.12356665e-02  2.84196273e-03 -1.87701732e-02 -2.53793895e-02
  6.02760501e-02  6.48584366e-02 -5.81610985e-02  1.36061728e-01
 -1.13502264e-01  8.68010335e-03  9.02378261e-02 -1.6003114

## 71. データセットの読み込み

[General Language Understanding Evaluation (GLUE)](https://gluebenchmark.com/) ベンチマークで配布されている[Stanford Sentiment Treebank (SST)](https://dl.fbaipublicfiles.com/glue/data/SST-2.zip) をダウンロードし、訓練セット（train.tsv）と開発セット（dev.tsv）のテキストと極性ラベルと読み込み、全てのテキストをトークンID列に変換せよ。このとき、単語埋め込みの語彙でカバーされていない単語は無視し、トークン列に含めないことにせよ。また、テキストの全トークンが単語埋め込みの語彙に含まれておらず、空のトークン列となってしまう事例は、訓練セットおよび開発セットから削除せよ（このため、第7章の実験で得られた正解率と比較できなくなることに注意せよ）。

事例の表現方法は任意でよいが、例えば"contains no wit , only labored gags"がネガティブに分類される事例は、次のような辞書オブジェクトで表現すればよい。

```
{'text': 'contains no wit , only labored gags',
 'label': tensor([0.]),
 'input_ids': tensor([ 3475,    87, 15888,    90, 27695, 42637])}
```

この例では、`text`はテキスト、`label`は分類ラベル（ポジティブなら`tensor([1.])`、ネガティブなら`tensor([0.])`）、`input_ids`はテキストのトークン列をID列で表現している。

In [6]:
train_data = pd.read_csv(f"{SST2_PATH}/train.tsv", sep="\t")
dev_data = pd.read_csv(f"{SST2_PATH}/dev.tsv", sep="\t")


def create_dict_objects(df):
    dict_objects = []
    for _, row in df.iterrows():
        dict_object = dict()
        dict_object["text"] = row["sentence"]
        dict_object["label"] = torch.tensor([row["label"]], dtype=torch.float32)
        dict_object["input_ids"] = torch.tensor([token_to_idx[word] for word in row["sentence"].split() if token_to_idx.get(word, False)])
        if len(dict_object["input_ids"]) != 0:
            dict_objects.append(dict_object)
    return dict_objects

train = create_dict_objects(train_data)
dev = create_dict_objects(dev_data)


print(f"train_data: {len(train_data)}")
print(f"dev_data: {len(dev_data)}")
print(f"train: {len(train)}")
print(f"dev: {len(dev)}")
print(train[0])


train_data: 67349
dev_data: 872
train: 66650
dev: 872
{'text': 'hide new secretions from the parental units ', 'label': tensor([0.]), 'input_ids': tensor([  5785,     66, 113845,     18,     12,  15095,   1594])}


## 72. Bag of wordsモデルの構築

単語埋め込みの平均ベクトルでテキストの特徴ベクトルを表現し、重みベクトルとの内積でポジティブ及びネガティブを分類するニューラルネットワーク（ロジスティック回帰モデル）を設計せよ。

In [7]:

class LogisticRegression(nn.Module):
    def __init__(self, input_size, output_size, embedding, freeze=True):
        super().__init__()
        self.layer = nn.Linear(input_size, output_size)
        self.sigmoid = nn.Sigmoid()
        self.embedding = nn.Embedding.from_pretrained(embedding, freeze=freeze)

    def forward(self, input_ids):
        emb = self.embedding(input_ids)
        x = torch.mean(emb, dim=1)
        y = self.layer(x)
        z = self.sigmoid(y)
        return z

class MyDataset(Dataset):
    def __init__(self, dict_objects):
        super().__init__()
        self.dict_objects = dict_objects

    def __getitem__(self, index):
        input_ids = self.dict_objects[index]["input_ids"]
        text = self.dict_objects[index]["text"]
        label = self.dict_objects[index]["label"]
        return input_ids, text, label
    
    def __len__(self):
        return len(self.dict_objects)



## 73. モデルの学習

問題72で設計したモデルの重みベクトルを訓練セット上で学習せよ。ただし、学習中は単語埋め込み行列の値を固定せよ（単語埋め込み行列のファインチューニングは行わない）。また、学習時に損失値を表示するなど、学習の進捗状況をモニタリングできるようにせよ。

In [24]:
epochs = 20
lr = 0.001
emb_ts = torch.tensor(EMB)
model = LogisticRegression(vector_size, 1, embedding=emb_ts).to(device)
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(),lr=lr)
batch_size = 1

train_ds = MyDataset(train)
dev_ds = MyDataset(dev)
train_dl = DataLoader(train_ds, batch_size, shuffle=True)
dev_dl = DataLoader(dev_ds, batch_size, shuffle=False)

loss_hist = []

for epoch in range(epochs):
    epoch_loss = 0
    for x, _, y in train_dl:
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = loss_fn(pred, y)

        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_dl)
    print(f"Epoch {epoch+1} Loss {avg_loss:4f}")
    if epoch > 1:
        if avg_loss > loss_hist[epoch-1]:
            break
    loss_hist.append(avg_loss)



Epoch 1 Loss 0.443147
Epoch 2 Loss 0.385306
Epoch 3 Loss 0.376951
Epoch 4 Loss 0.373422
Epoch 5 Loss 0.371572
Epoch 6 Loss 0.370304
Epoch 7 Loss 0.369645
Epoch 8 Loss 0.368990
Epoch 9 Loss 0.368463
Epoch 10 Loss 0.368265
Epoch 11 Loss 0.368083
Epoch 12 Loss 0.367959
Epoch 13 Loss 0.367742
Epoch 14 Loss 0.367699
Epoch 15 Loss 0.367528
Epoch 16 Loss 0.367428
Epoch 17 Loss 0.367397
Epoch 18 Loss 0.367396
Epoch 19 Loss 0.367400


## 74. モデルの評価

問題73で学習したモデルの開発セットにおける正解率を求めよ。

In [25]:
def cal_acc(model, dev_dl, device):
    correct = 0
    total = 0
    with torch.no_grad():
        for x, _, y in dev_dl:
            x = x.to(device)
            y = y.to(device)

            outputs = model(x)

            preds = (outputs > 0.5).float()

            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total
print(cal_acc(model, dev_dl, device))


0.7993119266055045


## 75. パディング

複数の事例が与えられたとき、これらをまとめて一つのテンソル・オブジェクトで表現する関数`collate`を実装せよ。与えられた複数の事例のトークン列の長さが異なるときは、トークン列の長さが最も長いものに揃え、0番のトークンIDでパディングをせよ。さらに、トークン列の長さが長いものから順に、事例を並び替えよ。

例えば、訓練データセットの冒頭の4事例が次のように表されているとき、

```
[{'text': 'hide new secretions from the parental units',
  'label': tensor([0.]),
  'input_ids': tensor([  5785,     66, 113845,     18,     12,  15095,   1594])},
 {'text': 'contains no wit , only labored gags',
  'label': tensor([0.]),
  'input_ids': tensor([ 3475,    87, 15888,    90, 27695, 42637])},
 {'text': 'that loves its characters and communicates something rather beautiful about human nature',
  'label': tensor([1.]),
  'input_ids': tensor([    4,  5053,    45,  3305, 31647,   348,   904,  2815,    47,  1276,  1964])},
 {'text': 'remains utterly satisfied to remain the same throughout',
  'label': tensor([0.]),
  'input_ids': tensor([  987, 14528,  4941,   873,    12,   208,   898])}]
```

`collate`関数を通した結果は以下のようになることが想定される。

```
{'input_ids': tensor([
    [     4,   5053,     45,   3305,  31647,    348,    904,   2815,     47,   1276,   1964],
    [  5785,     66, 113845,     18,     12,  15095,   1594,      0,      0,      0,      0],
    [   987,  14528,   4941,    873,     12,    208,    898,      0,      0,      0,      0],
    [  3475,     87,  15888,     90,  27695,  42637,      0,      0,      0,      0,      0]]),
 'label': tensor([
    [1.],
    [0.],
    [0.],
    [0.]])}
```


In [26]:
def collate(data_list):
    tmp = []
    
    for data in data_list:
        input_ids = data["input_ids"]
        label = data["label"]
        tmp.append([len(input_ids), input_ids, label])
    tmp = sorted(tmp,reverse=True)

    return {"input_ids":pad_sequence([data[1] for data in tmp], batch_first=True), "label":torch.tensor([data[2] for data in tmp])}

print(collate(train[:3]))


{'input_ids': tensor([[     4,   5053,     45,   3305,  31647,    348,    904,   2815,     47,
           1276,   1964],
        [  5785,     66, 113845,     18,     12,  15095,   1594,      0,      0,
              0,      0],
        [  3475,     87,  15888,     90,  27695,  42637,      0,      0,      0,
              0,      0]]), 'label': tensor([1., 0., 0.])}


## 76. ミニバッチ学習

問題75のパディングの処理を活用して、ミニバッチでモデルを学習せよ。また、学習したモデルの開発セットにおける正解率を求めよ。

In [32]:
def collate_fn(batch):
    input_ids, text, targets = list(zip(*batch))
    
    input_ids = pad_sequence(input_ids, batch_first=True)
    targets = torch.stack(targets)

    return input_ids, text, targets

epochs = 20
lr = 0.01
model_minibatch = LogisticRegression(vector_size, 1, embedding=emb_ts).to(device)
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model_minibatch.parameters(),lr=lr)
batch_size = 256

train_ds = MyDataset(train)
dev_ds = MyDataset(dev)
train_dl = DataLoader(train_ds, batch_size, shuffle=True, collate_fn=collate_fn, num_workers=2)
dev_dl = DataLoader(dev_ds, batch_size, shuffle=False, collate_fn=collate_fn, num_workers=2)

loss_hist = []

for epoch in range(epochs):
    epoch_loss = 0
    for x, _, y in train_dl:
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        pred = model_minibatch(x)

        loss = loss_fn(pred, y)

        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_dl)
    print(f"Epoch {epoch+1} Loss {avg_loss:4f}")
    if epoch > 1:
        if avg_loss > loss_hist[epoch-1]:
            break
    loss_hist.append(avg_loss)


print("=============")
print(cal_acc(model_minibatch, dev_dl, device))


Epoch 1 Loss 0.643911
Epoch 2 Loss 0.584230
Epoch 3 Loss 0.545656
Epoch 4 Loss 0.517865
Epoch 5 Loss 0.496905
Epoch 6 Loss 0.481956
Epoch 7 Loss 0.469514
Epoch 8 Loss 0.458969
Epoch 9 Loss 0.451045
Epoch 10 Loss 0.444473
Epoch 11 Loss 0.438598
Epoch 12 Loss 0.434062
Epoch 13 Loss 0.429670
Epoch 14 Loss 0.425656
Epoch 15 Loss 0.422261
Epoch 16 Loss 0.420170
Epoch 17 Loss 0.417559
Epoch 18 Loss 0.414850
Epoch 19 Loss 0.413269
Epoch 20 Loss 0.411356
0.7924311926605505


## 77. GPU上での学習

問題76のモデル学習をGPU上で実行せよ。また、学習したモデルの開発セットにおける正解率を求めよ。

## 78. 単語埋め込みのファインチューニング

問題77の学習において、単語埋め込みのパラメータも同時に更新するファインチューニングを導入せよ。また、学習したモデルの開発セットにおける正解率を求めよ。

In [33]:
def collate_fn(batch):
    input_ids, text, targets = list(zip(*batch))
    
    input_ids = pad_sequence(input_ids, batch_first=True)
    targets = torch.stack(targets)

    return input_ids, text, targets

epochs = 20
lr = 0.01
model_minibatch = LogisticRegression(vector_size, 1, embedding=emb_ts, freeze=False).to(device)
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model_minibatch.parameters(),lr=lr)
batch_size = 256

train_ds = MyDataset(train)
dev_ds = MyDataset(dev)
train_dl = DataLoader(train_ds, batch_size, shuffle=True, collate_fn=collate_fn, num_workers=2)
dev_dl = DataLoader(dev_ds, batch_size, shuffle=False, collate_fn=collate_fn, num_workers=2)

loss_hist = []

for epoch in range(epochs):
    epoch_loss = 0
    for x, _, y in train_dl:
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        pred = model_minibatch(x)

        loss = loss_fn(pred, y)

        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_dl)
    print(f"Epoch {epoch+1} Loss {avg_loss:4f}")
    if epoch > 1:
        if avg_loss > loss_hist[epoch-1]:
            break
    loss_hist.append(avg_loss)


print("=============")
print(cal_acc(model_minibatch, dev_dl, device))


Epoch 1 Loss 0.376383
Epoch 2 Loss 0.229489
Epoch 3 Loss 0.198490
Epoch 4 Loss 0.187634
Epoch 5 Loss 0.174182
Epoch 6 Loss 0.167165
Epoch 7 Loss 0.161019
Epoch 8 Loss 0.158071
Epoch 9 Loss 0.154938
Epoch 10 Loss 0.152154
Epoch 11 Loss 0.150179
Epoch 12 Loss 0.148795
Epoch 13 Loss 0.146303
Epoch 14 Loss 0.144546
Epoch 15 Loss 0.144425
Epoch 16 Loss 0.143328
Epoch 17 Loss 0.142900
Epoch 18 Loss 0.141193
Epoch 19 Loss 0.139649
Epoch 20 Loss 0.139314
0.7729357798165137


## 79. アーキテクチャの変更

ニューラルネットワークのアーキテクチャを自由に変更し、モデルを学習せよ。また、学習したモデルの開発セットにおける正解率を求めよ。例えば、テキストの特徴ベクトル（単語埋め込みの平均ベクトル）に対して多層のニューラルネットワークを通したり、畳み込みニューラルネットワーク（CNN; Convolutional Neural Network）や再帰型ニューラルネットワーク（RNN; Recurrent Neural Network）などのモデルの学習に挑戦するとよい。

In [ ]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=100, num_layers=2, output_size=1, embedding=None, freeze=True):
        super().__init__()
        self.linear= nn.Linear(hidden_size, output_size)
        self.sigmoid = nn.Sigmoid()
        self.embedding = nn.Embedding.from_pretrained(embedding, freeze=freeze)
        self.rnn = nn.RNN(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)

    def forward(self, input_ids):
        emb = self.embedding(input_ids)
        x = torch.mean(emb, dim=1)
        _, hidden = self.rnn(x)
        out = hidden[-1, :]
        y = self.linear(out)
        z = self.sigmoid(y)
        return z

class LSTM(nn.Module):
    def __init__(self, input_size, hidden_size=100, num_layers=2, output_size=1, embedding=None, freeze=True):
        super().__init__()
        self.linear= nn.Linear(hidden_size, output_size)
        self.sigmoid = nn.Sigmoid()
        self.embedding = nn.Embedding.from_pretrained(embedding, freeze=freeze)
        self.rnn = nn.RNN(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)

    def forward(self, input_ids):
        emb = self.embedding(input_ids)
        x = torch.mean(emb, dim=1)
        _, hidden = self.rnn(x)
        out = hidden[-1, :, :]
        y = self.linear(out)
        z = self.sigmoid(y)
        return z
    

In [21]:
def collate_fn(batch):
    input_ids, text, targets = list(zip(*batch))
    
    input_ids = pad_sequence(input_ids, batch_first=True)
    targets = torch.stack(targets)

    return input_ids, text, targets

epochs = 20
lr = 0.01
emb_ts = torch.tensor(EMB)
rnn = RNN(input_size=vector_size, hidden_size=100, num_layers=1, output_size=1, embedding=emb_ts, freeze=False).to(device)
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(rnn.parameters(),lr=lr)
batch_size = 256

train_ds = MyDataset(train)
dev_ds = MyDataset(dev)
train_dl = DataLoader(train_ds, batch_size, shuffle=True, collate_fn=collate_fn, num_workers=2)
dev_dl = DataLoader(dev_ds, batch_size, shuffle=False, collate_fn=collate_fn, num_workers=2)

loss_hist = []

for epoch in range(epochs):
    epoch_loss = 0
    for x, _, y in train_dl:
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        pred = rnn(x)
        loss = loss_fn(pred, y)

        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_dl)
    print(f"Epoch {epoch+1} Loss {avg_loss:4f}")
    if epoch > 1:
        if avg_loss > loss_hist[epoch-1]:
            break
    loss_hist.append(avg_loss)


print("=============")
print(cal_acc(rnn, dev_dl, device))


OutOfMemoryError: CUDA out of memory. Tried to allocate 3.35 GiB. GPU 0 has a total capacity of 23.64 GiB of which 2.58 GiB is free. Including non-PyTorch memory, this process has 21.06 GiB memory in use. Of the allocated memory 20.15 GiB is allocated by PyTorch, and 16.55 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)